# Week 1 — Day 4: Tokens and Conversation State

## Goals

- Understand what tokens are
- Encode and decode text with tiktoken
- Understand why token count matters
- Understand that separate LLM calls do not automatically share state
- Learn how conversation state can be preserved
- Use the modern OpenAI Responses API

## Official Documentation

- OpenAI Conversation State:
  https://developers.openai.com/api/docs/guides/conversation-state

- OpenAI Text Generation:
  https://developers.openai.com/api/docs/guides/text

- OpenAI tiktoken:
  https://github.com/openai/tiktoken

In [1]:
import tiktoken

encoding = tiktoken.encoding_for_model("gpt-4.1-mini")

text = "Hi, my name is Siva and I am learning LLM engineering."

tokens = encoding.encode(text)

tokens

[12194,
 11,
 922,
 1308,
 382,
 336,
 5638,
 326,
 357,
 939,
 7524,
 451,
 19641,
 16411,
 13]

In [2]:
len(tokens) 

15

In [3]:
for token_id in tokens:
    token_text = encoding.decode([token_id])
    print(f"{token_id} -> {repr(token_text)}")

12194 -> 'Hi'
11 -> ','
922 -> ' my'
1308 -> ' name'
382 -> ' is'
336 -> ' S'
5638 -> 'iva'
326 -> ' and'
357 -> ' I'
939 -> ' am'
7524 -> ' learning'
451 -> ' L'
19641 -> 'LM'
16411 -> ' engineering'
13 -> '.'


## Stateless LLM Calls and Conversation Memory

An LLM does not automatically remember previous independent API calls.

To create conversational memory, previous messages or response state must be provided again.

## Stateless Calls

Two separate API calls do not automatically share the conversation unless we explicitly preserve the state.

In [8]:
import os

from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

load_dotenv("../.env", override=True)

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)
print(type(client))



<class 'openai.OpenAI'>


## Experiment 1 — two independent calls

In [9]:


first_response = client.responses.create(
    model="gpt-5-mini",
    input="My name is Siva."
)

display(Markdown(first_response.output_text))

Nice to meet you, Siva. How can I help you today?

In [10]:
second_response = client.responses.create(
    model="gpt-5-mini",
    input="What is my name?"
)

display(Markdown(second_response.output_text))

I don't know—I don't have access to your personal info unless you tell me. What would you like me to call you? If you tell me your name or a nickname, I can use it for this conversation.

## Experiment 2 — preserve conversation state

In [13]:
third_response = client.responses.create(
    model="gpt-5-mini",
    input="My name is Siva."
)


In [14]:
fourth_response = client.responses.create(
    model="gpt-5-mini",
    previous_response_id=third_response.id,
    input="What is my name?"
)

display(Markdown(fourth_response.output_text))

Your name is Siva.

# Day 4 — What I Learned

## Tokenization

- LLMs process tokens rather than words directly.
- One word does not necessarily equal one token.
- Tokens can represent complete words, parts of words, spaces, punctuation, or other text fragments.
- `tiktoken.encoding_for_model()` selects an appropriate tokenizer for an OpenAI model.
- `encode()` converts text into token IDs.
- `decode()` converts token IDs back into text.
- Token count matters because it affects context usage and API cost.

## Conversation State

- A separate API request does not automatically know about a previous request.
- What appears to be LLM memory is often conversation context managed by the application.
- OpenAI's Responses API can continue a conversation using `previous_response_id`.
- Application memory and model intelligence are different concepts.
- Later, persistent memory can involve databases, Redis, LangGraph checkpoints, vector stores, or other storage systems.

## Engineering Takeaway

I should remember the architecture rather than memorize SDK syntax:

User input
→ application
→ context/state
→ model API
→ response
→ application stores or forwards relevant state